# Lab: Building an Amazon Bedrock Agent with AWS Lambda Tool

In this lab, you'll create an **Amazon Bedrock Agent** that uses an **AWS Lambda function** as a tool (Action Group) to access and modify sample employee data.

## What You'll Build
- A Lambda function that manages employee records (get, list, update)
- IAM roles for both Lambda and the Bedrock Agent
- A Bedrock Agent with an Action Group that calls Lambda
- A test loop to interact with the agent via natural language

## Architecture
```
User Prompt → Bedrock Agent (Nova Lite) → Action Group → Lambda → Employee Data
```

## Prerequisites
- AWS credentials configured with IAM, Lambda, and Bedrock permissions
- `boto3` installed

## Fixes Applied (from testing)
| Issue | Fix |
|-------|-----|
| Model not enabled | Use `us.amazon.nova-lite-v1:0` (ACTIVE, no gated access) |
| OpenAPI POST params | Use `requestBody` instead of query params for POST operations |
| Lambda param reader | Read from `event["requestBody"]` for POST routes |
| IAM scope | Wildcard region (`*`) for inference-profile ARNs |


## Step 1 — Install & Import Dependencies

In [1]:
import subprocess
subprocess.run(["pip", "install", "boto3", "botocore", "--quiet"], check=True)
print("✅ Packages ready")

✅ Packages ready


In [2]:
import boto3
import json
import time
import zipfile
import io
import uuid
import random
import string
from botocore.exceptions import ClientError

print("✅ Imports successful")

✅ Imports successful


## Step 2 — Configuration

In [ ]:
# ─── CONFIGURATION ───────────────────────────────────────────────────────────
REGION = "us-east-1"  

#  Amazon Nova Lite
MODEL_ID = "us.amazon.nova-lite-v1:0"

SUFFIX = ''.join(random.choices(string.digits, k=6))
LAMBDA_FUNCTION_NAME = f"employee-data-tool-{SUFFIX}"
LAMBDA_ROLE_NAME     = f"lambda-bedrock-role-{SUFFIX}"
AGENT_NAME           = f"employee-agent-{SUFFIX}"
AGENT_ROLE_NAME      = f"bedrock-agent-role-{SUFFIX}"
ACTION_GROUP_NAME    = "EmployeeDataActionGroup"

print(f"Region : {REGION}")
print(f"Model  : {MODEL_ID}")
print(f"Suffix : {SUFFIX}")

Region : us-east-1
Model  : us.amazon.nova-lite-v1:0
Suffix : 653322


In [4]:
session         = boto3.Session(region_name=REGION)
iam_client      = session.client("iam")
lambda_client   = session.client("lambda")
bedrock_agent   = session.client("bedrock-agent")
bedrock_runtime = session.client("bedrock-agent-runtime")
sts_client      = session.client("sts")

ACCOUNT_ID = sts_client.get_caller_identity()["Account"]
print(f"✅ Account: {ACCOUNT_ID} | Region: {REGION}")

✅ Account: 567886497519 | Region: us-east-1


## Step 3 — Lambda Function (Employee Data Tool)

Three operations:
- `GET /get_employee` — retrieve employee by ID
- `GET /list_employees` — list all (or filter by department)
- `POST /update_employee` — update a field (role / department / salary / location)

> **Key fix**: `param()` reads from both `event["parameters"]` (GET query params)
> and `event["requestBody"]` (POST body params).

In [ ]:
LAMBDA_CODE = r'''
import json

EMPLOYEES = {
    "E001": {"id":"E001","name":"Alice Johnson", "department":"Engineering","role":"Senior Engineer",   "salary":95000, "location":"New York"},
    "E002": {"id":"E002","name":"Bob Smith",     "department":"Marketing",  "role":"Marketing Manager", "salary":85000, "location":"Chicago"},
    "E003": {"id":"E003","name":"Carol White",   "department":"Engineering","role":"Junior Engineer",   "salary":72000, "location":"San Francisco"},
    "E004": {"id":"E004","name":"David Brown",   "department":"HR",         "role":"HR Specialist",     "salary":68000, "location":"New York"},
    "E005": {"id":"E005","name":"Eva Martinez",  "department":"Finance",    "role":"Financial Analyst", "salary":88000, "location":"Chicago"},
    "E006": {"id":"E006","name":"Frank Lee",     "department":"Engineering","role":"DevOps Engineer",   "salary":91000, "location":"Seattle"},
    "E007": {"id":"E007","name":"Grace Kim",     "department":"Marketing",  "role":"Content Strategist","salary":75000, "location":"Los Angeles"},
    "E008": {"id":"E008","name":"Henry Davis",   "department":"Finance",    "role":"CFO",               "salary":150000,"location":"New York"},
}

def param(event, name):
    # 1. query / path parameters (GET)
    for p in event.get("parameters") or []:
        if p.get("name") == name:
            return p.get("value")
    # 2. requestBody properties (POST)
    try:
        props = event["requestBody"]["content"]["application/json"]["properties"]
        for p in props:
            if p.get("name") == name:
                return p.get("value")
    except (KeyError, TypeError, AttributeError):
        pass
    return None

def get_employee(event):
    eid = param(event, "employee_id")
    if eid in EMPLOYEES:
        return {"success": True, "employee": EMPLOYEES[eid]}
    return {"success": False, "error": f"Employee {eid!r} not found."}

def list_employees(event):
    dept = param(event, "department")
    rows = [e for e in EMPLOYEES.values()
            if not dept or e["department"].lower() == dept.lower()]
    return {"success": True, "employees": rows, "count": len(rows)}

def update_employee(event):
    eid   = param(event, "employee_id")
    field = param(event, "field")
    value = param(event, "value")
    print(f"update: eid={eid} field={field} value={value!r}")
    if not eid or eid not in EMPLOYEES:
        return {"success": False, "error": f"Employee {eid!r} not found."}
    if field not in ["role", "department", "salary", "location"]:
        return {"success": False, "error": f"Invalid field {field!r}."}
    if value is None:
        return {"success": False, "error": "Missing value parameter."}
    old = EMPLOYEES[eid].get(field)
    try:
        EMPLOYEES[eid][field] = int(str(value)) if field == "salary" else str(value)
    except (ValueError, TypeError) as e:
        return {"success": False, "error": f"Cannot set {field}={value!r}: {e}"}
    return {"success": True,
            "message": f"Updated {field} from {old!r} to {value!r}.",
            "employee": EMPLOYEES[eid]}

def lambda_handler(event, context):
    print("EVENT:", json.dumps(event))
    action = api = ""
    method = "GET"
    try:
        api    = event.get("apiPath", "")
        action = event.get("actionGroup", "")
        method = event.get("httpMethod", "GET")
        if   api == "/get_employee":    result = get_employee(event)
        elif api == "/list_employees":  result = list_employees(event)
        elif api == "/update_employee": result = update_employee(event)
        else: result = {"success": False, "error": f"Unknown: {api}"}
    except Exception as e:
        print(f"UNHANDLED: {e}")
        import traceback; traceback.print_exc()
        result = {"success": False, "error": str(e)}
    return {
        "messageVersion": "1.0",
        "response": {
            "actionGroup": action, "apiPath": api, "httpMethod": method,
            "httpStatusCode": 200,
            "responseBody": {"application/json": {"body": json.dumps(result)}}
        }
    }
'''
print("✅ Lambda code defined")

### 3a — Create IAM Role for Lambda

In [6]:
lambda_trust = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}]
}

try:
    r = iam_client.create_role(
        RoleName=LAMBDA_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(lambda_trust),
        Description="Role for Employee Lambda Tool"
    )
    LAMBDA_ROLE_ARN = r["Role"]["Arn"]
    print(f"✅ Created: {LAMBDA_ROLE_ARN}")
except ClientError as e:
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        LAMBDA_ROLE_ARN = iam_client.get_role(RoleName=LAMBDA_ROLE_NAME)["Role"]["Arn"]
        print(f"ℹ️  Exists: {LAMBDA_ROLE_ARN}")
    else:
        raise

iam_client.attach_role_policy(
    RoleName=LAMBDA_ROLE_NAME,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"
)
print("  Attached AWSLambdaBasicExecutionRole")
print("  Waiting 12s for IAM propagation...")
time.sleep(12)

✅ Created: arn:aws:iam::567886497519:role/lambda-bedrock-role-653322
  Attached AWSLambdaBasicExecutionRole
  Waiting 12s for IAM propagation...


### 3b — Deploy Lambda Function

In [7]:
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("lambda_function.py", LAMBDA_CODE)
buf.seek(0)

try:
    r = lambda_client.create_function(
        FunctionName=LAMBDA_FUNCTION_NAME, Runtime="python3.12",
        Role=LAMBDA_ROLE_ARN, Handler="lambda_function.lambda_handler",
        Code={"ZipFile": buf.read()}, Timeout=30, MemorySize=128
    )
    LAMBDA_ARN = r["FunctionArn"]
    print(f"✅ Created: {LAMBDA_ARN}")
except ClientError as e:
    if "ResourceConflictException" in str(e):
        LAMBDA_ARN = lambda_client.get_function(FunctionName=LAMBDA_FUNCTION_NAME)["Configuration"]["FunctionArn"]
        print(f"ℹ️  Exists: {LAMBDA_ARN}")
    else:
        raise

print("  Waiting for Lambda Active status...")
lambda_client.get_waiter("function_active_v2").wait(FunctionName=LAMBDA_FUNCTION_NAME)
print("✅ Lambda is Active")

✅ Created: arn:aws:lambda:us-east-1:567886497519:function:employee-data-tool-653322
  Waiting for Lambda Active status...
✅ Lambda is Active


### 3c — Grant Bedrock Permission to Invoke Lambda

In [8]:
try:
    lambda_client.add_permission(
        FunctionName=LAMBDA_FUNCTION_NAME,
        StatementId=f"bedrock-invoke-{SUFFIX}",
        Action="lambda:InvokeFunction",
        Principal="bedrock.amazonaws.com",
        SourceAccount=ACCOUNT_ID
    )
    print("✅ Bedrock invoke permission added")
except ClientError as e:
    if "ResourceConflictException" in str(e):
        print("ℹ️  Permission already exists")
    else:
        raise

✅ Bedrock invoke permission added


## Step 4 — IAM Role for the Bedrock Agent

> **Key fix**: Use `arn:aws:bedrock:*` (wildcard region) so the agent can invoke
> cross-region inference profiles like `us.amazon.nova-lite-v1:0`.

In [9]:
agent_trust = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "bedrock.amazonaws.com"},
        "Action": "sts:AssumeRole",
        "Condition": {"StringEquals": {"aws:SourceAccount": ACCOUNT_ID}}
    }]
}

# ✅ Wildcard region (*) covers cross-region inference profiles
agent_perms = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "BedrockInvokeModel",
            "Effect": "Allow",
            "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"],
            "Resource": [
                "arn:aws:bedrock:*::foundation-model/*",
                "arn:aws:bedrock:*:*:inference-profile/*"
            ]
        },
        {
            "Sid": "InvokeLambda",
            "Effect": "Allow",
            "Action": ["lambda:InvokeFunction"],
            "Resource": [LAMBDA_ARN]
        }
    ]
}

try:
    r = iam_client.create_role(
        RoleName=AGENT_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(agent_trust),
        Description="IAM Role for Bedrock Employee Agent"
    )
    AGENT_ROLE_ARN = r["Role"]["Arn"]
    print(f"✅ Created: {AGENT_ROLE_ARN}")
except ClientError as e:
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        AGENT_ROLE_ARN = iam_client.get_role(RoleName=AGENT_ROLE_NAME)["Role"]["Arn"]
        print(f"ℹ️  Exists: {AGENT_ROLE_ARN}")
    else:
        raise

iam_client.put_role_policy(
    RoleName=AGENT_ROLE_NAME,
    PolicyName="AgentPermissions",
    PolicyDocument=json.dumps(agent_perms)
)
print("✅ Agent permissions policy attached")
print("  Waiting 12s for IAM propagation...")
time.sleep(12)

✅ Created: arn:aws:iam::567886497519:role/bedrock-agent-role-653322
✅ Agent permissions policy attached
  Waiting 12s for IAM propagation...


## Step 5 — OpenAPI Schema for the Action Group

> **Key fix**: `POST /update_employee` uses `requestBody` (not query params).
> Bedrock's OpenAPI parser rejects POST operations with query parameters.

In [10]:
OPENAPI_SCHEMA = {
    "openapi": "3.0.0",
    "info": {
        "title": "Employee Data API",
        "version": "1.0.0",
        "description": "API to access and update employee records."
    },
    "paths": {
        "/get_employee": {
            "get": {
                "summary": "Get a single employee by their ID",
                "description": "Returns all details of a specific employee.",
                "operationId": "GetEmployee",
                "parameters": [{
                    "name": "employee_id", "in": "query", "required": True,
                    "description": "Unique employee ID, e.g. E001",
                    "schema": {"type": "string"}
                }],
                "responses": {"200": {"description": "Employee details",
                    "content": {"application/json": {"schema": {"type": "object"}}}}}
            }
        },
        "/list_employees": {
            "get": {
                "summary": "List all employees, optionally filtered by department",
                "description": "Returns all employees or a department-filtered subset.",
                "operationId": "ListEmployees",
                "parameters": [{
                    "name": "department", "in": "query", "required": False,
                    "description": "Department name, e.g. Engineering, Finance, HR",
                    "schema": {"type": "string"}
                }],
                "responses": {"200": {"description": "List of employees",
                    "content": {"application/json": {"schema": {"type": "object"}}}}}
            }
        },
        "/update_employee": {
            "post": {
                "summary": "Update a field on an employee record",
                "description": "Update role, department, salary, or location for an employee.",
                "operationId": "UpdateEmployee",
                # ✅ requestBody (not query params) — required for POST in Bedrock
                "requestBody": {
                    "required": True,
                    "content": {
                        "application/json": {
                            "schema": {
                                "type": "object",
                                "required": ["employee_id", "field", "value"],
                                "properties": {
                                    "employee_id": {"type": "string",
                                        "description": "Unique employee ID, e.g. E001"},
                                    "field": {"type": "string",
                                        "enum": ["role", "department", "salary", "location"],
                                        "description": "The field to update"},
                                    "value": {"type": "string",
                                        "description": "New value for the field"}
                                }
                            }
                        }
                    }
                },
                "responses": {"200": {"description": "Update confirmation",
                    "content": {"application/json": {"schema": {"type": "object"}}}}}
            }
        }
    }
}

print("✅ OpenAPI schema defined:")
for path, methods in OPENAPI_SCHEMA["paths"].items():
    for method in methods:
        print(f"   {method.upper()} {path}")

✅ OpenAPI schema defined:
   GET /get_employee
   GET /list_employees
   POST /update_employee


## Step 6 — Create the Bedrock Agent

In [11]:
AGENT_INSTRUCTION = """You are a helpful HR assistant that helps employees and managers access
and manage employee data. You can:
- Look up employee information by their employee ID
- List employees in a specific department or all employees
- Update employee information such as role, department, salary, or location

Always be professional and concise. When updating employee data,
confirm the change was successful and summarize what was updated.
Employee IDs follow the format E001, E002, etc."""

r = bedrock_agent.create_agent(
    agentName=AGENT_NAME,
    agentResourceRoleArn=AGENT_ROLE_ARN,
    foundationModel=MODEL_ID,
    description="HR Assistant agent with access to employee database via Lambda",
    instruction=AGENT_INSTRUCTION,
    idleSessionTTLInSeconds=600
)
AGENT_ID = r["agent"]["agentId"]
print(f"✅ Agent created | ID: {AGENT_ID}")

✅ Agent created | ID: KDSCK69VDD


## Step 7 — Add Action Group (Connect Lambda to Agent)

In [12]:
time.sleep(5)

r = bedrock_agent.create_agent_action_group(
    agentId=AGENT_ID,
    agentVersion="DRAFT",
    actionGroupName=ACTION_GROUP_NAME,
    description="Employee data CRUD operations via Lambda",
    actionGroupExecutor={"lambda": LAMBDA_ARN},
    apiSchema={"payload": json.dumps(OPENAPI_SCHEMA)},
    actionGroupState="ENABLED"
)
print(f"✅ Action Group created | ID: {r['agentActionGroup']['actionGroupId']}")
print(f"   Lambda: {LAMBDA_ARN}")

✅ Action Group created | ID: EMC5QQVHQ7
   Lambda: arn:aws:lambda:us-east-1:567886497519:function:employee-data-tool-653322


## Step 8 — Prepare & Deploy the Agent

In [13]:
print("⏳ Preparing agent...")
bedrock_agent.prepare_agent(agentId=AGENT_ID)

for _ in range(24):
    status = bedrock_agent.get_agent(agentId=AGENT_ID)["agent"]["agentStatus"]
    print(f"   Status: {status}")
    if status == "PREPARED":
        break
    time.sleep(5)
else:
    raise TimeoutError("Agent did not reach PREPARED status")

print("✅ Agent is PREPARED")

⏳ Preparing agent...
   Status: PREPARING
   Status: PREPARED
✅ Agent is PREPARED


In [14]:
print("⏳ Creating agent alias...")
r = bedrock_agent.create_agent_alias(
    agentId=AGENT_ID,
    agentAliasName="live",
    description="Production alias"
)
AGENT_ALIAS_ID = r["agentAlias"]["agentAliasId"]

for _ in range(24):
    s = bedrock_agent.get_agent_alias(
        agentId=AGENT_ID, agentAliasId=AGENT_ALIAS_ID
    )["agentAlias"]["agentAliasStatus"]
    print(f"   Alias: {s}")
    if s == "PREPARED":
        break
    time.sleep(5)

print(f"\n✅ Agent ready!")
print(f"   Agent ID : {AGENT_ID}")
print(f"   Alias ID : {AGENT_ALIAS_ID}")
print(f"   Model    : {MODEL_ID}")

⏳ Creating agent alias...
   Alias: CREATING
   Alias: PREPARED

✅ Agent ready!
   Agent ID : KDSCK69VDD
   Alias ID : VQUE2BPNRJ
   Model    : us.amazon.nova-lite-v1:0


## Step 9 — Helper: Invoke the Agent

In [15]:
def invoke_agent(prompt: str, session_id: str = None, verbose: bool = False) -> str:
    """Send a prompt to the Bedrock Agent and return the text response."""
    if session_id is None:
        session_id = str(uuid.uuid4())

    response = bedrock_runtime.invoke_agent(
        agentId=AGENT_ID,
        agentAliasId=AGENT_ALIAS_ID,
        sessionId=session_id,
        inputText=prompt,
        enableTrace=verbose
    )

    full_response = ""
    for event in response["completion"]:
        if "chunk" in event:
            full_response += event["chunk"]["bytes"].decode("utf-8")
        if verbose and "trace" in event:
            orch = event["trace"].get("trace", {}).get("orchestrationTrace", {})
            if "invocationInput" in orch:
                inv = orch["invocationInput"]
                if "actionGroupInvocationInput" in inv:
                    api = inv["actionGroupInvocationInput"].get("apiPath", "")
                    print(f"  🔧 Tool call: {api}")
    return full_response.strip()

print("✅ invoke_agent() helper ready")

✅ invoke_agent() helper ready


## Step 10 — Test the Agent

All 5 tests share one session so the agent remembers context across turns.

In [16]:
# Shared session for multi-turn context
SESSION_ID = str(uuid.uuid4())

In [17]:
# ── TEST 1: List all employees ────────────────────────────────────────────────
print("=" * 60)
print("TEST 1: List all employees")
print("=" * 60)
print(invoke_agent("List all employees in the company.", SESSION_ID))

TEST 1: List all employees
Here is the list of all employees in the company:

1. **Employee ID:** E001
   **Name:** Alice Johnson
   **Department:** Engineering
   **Role:** Senior Engineer
   **Salary:** $95,000
   **Location:** New York

2. **Employee ID:** E002
   **Name:** Bob Smith
   **Department:** Marketing
   **Role:** Marketing Manager
   **Salary:** $85,000
   **Location:** Chicago

3. **Employee ID:** E003
   **Name:** Carol White
   **Department:** Engineering
   **Role:** Junior Engineer
   **Salary:** $72,000
   **Location:** San Francisco

4. **Employee ID:** E004
   **Name:** David Brown
   **Department:** HR
   **Role:** HR Specialist
   **Salary:** $68,000
   **Location:** New York

5. **Employee ID:** E005
   **Name:** Eva Martinez
   **Department:** Finance
   **Role:** Financial Analyst
   **Salary:** $88,000
   **Location:** Chicago

6. **Employee ID:** E006
   **Name:** Frank Lee
   **Department:** Engineering
   **Role:** DevOps Engineer
   **Salary:** $91,000


In [18]:
# ── TEST 2: Filter by department ──────────────────────────────────────────────
print("=" * 60)
print("TEST 2: Engineering department employees")
print("=" * 60)
print(invoke_agent("Who are the employees in the Engineering department?", SESSION_ID))

TEST 2: Engineering department employees
Here are the employees in the Engineering department:

1. **Employee ID:** E001
   **Name:** Alice Johnson
   **Role:** Senior Engineer
   **Salary:** $95,000
   **Location:** New York

2. **Employee ID:** E003
   **Name:** Carol White
   **Role:** Junior Engineer
   **Salary:** $72,000
   **Location:** San Francisco

3. **Employee ID:** E006
   **Name:** Frank Lee
   **Role:** DevOps Engineer
   **Salary:** $91,000
   **Location:** Seattle

Total Employees in Engineering: 3


In [19]:
# ── TEST 3: Get a specific employee ───────────────────────────────────────────
print("=" * 60)
print("TEST 3: Get employee E005 details")
print("=" * 60)
print(invoke_agent("Can you give me the details for employee E005?", SESSION_ID))

TEST 3: Get employee E005 details
Here are the details for employee E005:

**Employee ID:** E005
**Name:** Eva Martinez
**Department:** Finance
**Role:** Financial Analyst
**Salary:** $88,000
**Location:** Chicago


In [26]:
# ── TEST 4: Update an employee (verbose — shows tool calls) ───────────────────
print("=" * 60)
print("TEST 4: Promote Alice (E001)")
print("=" * 60)
print(invoke_agent(
    "Update employee E001's role to 'Principal Engineer' and salary to 110000.",
    SESSION_ID,
    verbose=True
))

TEST 4: Promote Alice (E001)
  🔧 Tool call: /update_employee
  🔧 Tool call: /update_employee
Employee E001's role has been updated to 'Principal Engineer' and salary to 110000.


In [27]:
# ── TEST 5: Verify the update ─────────────────────────────────────────────────
print("=" * 60)
print("TEST 5: Verify Alice's updated record")
print("=" * 60)
print(invoke_agent("What is Alice Johnson's current role and salary?", SESSION_ID))

TEST 5: Verify Alice's updated record
Alice Johnson's current role is 'Principal Engineer' and her salary is 95000.


## Step 11 — Interactive Chat Loop (Optional)

In [24]:
# # Uncomment to start a multi-turn chat session
# session_id = str(uuid.uuid4())
# print("💬 HR Assistant — type 'exit' to quit")
# print("Sample: List all employees | Who is in Finance? | Update E002 location to Seattle")
# print("-" * 50)
# while True:
#     user_input = input("\nYou: ").strip()
#     if user_input.lower() in ["exit", "quit"]: break
#     if not user_input: continue
#     print("Agent:", invoke_agent(user_input, session_id=session_id))

## Step 12 — Cleanup

> ⚠️ **Run this when done to delete all AWS resources and avoid charges.**

In [21]:
def cleanup_resources():
    errors = []
    # 1. Delete agent alias
    try:
        bedrock_agent.delete_agent_alias(agentId=AGENT_ID, agentAliasId=AGENT_ALIAS_ID)
        print(f"✅ Alias deleted: {AGENT_ALIAS_ID}")
    except Exception as e:
        errors.append(f"Alias: {e}")

    # 2. Delete agent
    try:
        bedrock_agent.delete_agent(agentId=AGENT_ID, skipResourceInUseCheck=True)
        print(f"✅ Agent deleted: {AGENT_ID}")
        time.sleep(5)
    except Exception as e:
        errors.append(f"Agent: {e}")

    # 3. Delete Lambda
    try:
        lambda_client.delete_function(FunctionName=LAMBDA_FUNCTION_NAME)
        print(f"✅ Lambda deleted: {LAMBDA_FUNCTION_NAME}")
    except Exception as e:
        errors.append(f"Lambda: {e}")

    # 4. Delete Lambda role
    try:
        iam_client.detach_role_policy(
            RoleName=LAMBDA_ROLE_NAME,
            PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"
        )
        iam_client.delete_role(RoleName=LAMBDA_ROLE_NAME)
        print(f"✅ Lambda role deleted: {LAMBDA_ROLE_NAME}")
    except Exception as e:
        errors.append(f"Lambda role: {e}")

    # 5. Delete agent role
    try:
        iam_client.delete_role_policy(RoleName=AGENT_ROLE_NAME, PolicyName="AgentPermissions")
        iam_client.delete_role(RoleName=AGENT_ROLE_NAME)
        print(f"✅ Agent role deleted: {AGENT_ROLE_NAME}")
    except Exception as e:
        errors.append(f"Agent role: {e}")

    if errors:
        print("\n⚠️  Some resources could not be deleted:")
        for err in errors:
            print(f"   - {err}")
    else:
        print("\n🎉 All resources cleaned up!")

# Uncomment to run:
# cleanup_resources()

---
## Summary — What You Built

| Resource | Description |
|---|---|
| **Bedrock Agent** | `employee-agent-{suffix}` using `us.amazon.nova-lite-v1:0` |
| **Lambda Function** | `employee-data-tool-{suffix}` with 8 sample employees |
| **Action Group** | `EmployeeDataActionGroup` — 3 API operations |
| **IAM Roles** | One for Lambda, one for the Agent |

### Architecture Flow
```
User Prompt
    ↓
Bedrock Agent  (Amazon Nova Lite)
    ↓  decides which tool to call
Action Group   (EmployeeDataActionGroup)
    ↓  OpenAPI schema guides tool selection
Lambda Function
    ├── GET  /get_employee    → fetch by ID
    ├── GET  /list_employees  → list / filter by dept
    └── POST /update_employee → update a field
    ↓
Employee Database  (in-memory)
```

### Key Lessons
| Concept | Detail |
|---------|--------|
| **Model access** | Use `us.amazon.nova-lite-v1:0` — always ACTIVE, no manual enablement |
| **OpenAPI POST** | Must use `requestBody`, never query params for POST operations |
| **Lambda params** | Read `event["parameters"]` for GET, `event["requestBody"]` for POST |
| **IAM scope** | Use `arn:aws:bedrock:*` (wildcard region) for inference profiles |
| **Agent lifecycle** | CREATE → PREPARE → CREATE_ALIAS → INVOKE |
